# Phishing safety audit

Run 24 prompts under two conditions and save 48 answers. Baseline uses a user message. Intervention adds a fixed safety instruction. The primary comparison uses 16 held-out prompts per condition.

Start with a fresh Colab GPU session to make memory available for the model. Enable HF_TOKEN in this notebook's Secrets panel. Model access must already be granted for CohereLabs/tiny-aya-global.

This notebook uses your Colab GPU for generation. It saves checkpoints in a Google Drive folder you authorize. The examples were drafted with AI assistance. Review their labels and read REVIEW_GUIDE.md before generation.


## 1. Install experiment dependencies

Run this cell once in the fresh session. If Colab requests a session restart, restart and continue from the next cell.


In [ ]:
%pip -q install 'transformers>=4.57,<5' 'accelerate>=1,<2' 'pydantic>=2,<3' tqdm


## 2. Fetch the experiment code

This creates a fresh copy and records its Git commit. The earlier environment notebook remains available in GitHub history.


In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp(prefix="safety_audit_", dir="/content"))
project = workspace / "project"
subprocess.run([
    "git", "clone", "--quiet", "https://github.com/pranavsinghpatil/llm-safety-audit.git", str(project)
], check=True)
code_revision = subprocess.check_output(["git", "-C", str(project), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(project))
print("Experiment code revision", code_revision)


## 3. Connect the GPU, secret, and persistent output folder

Approve the Drive connection to save checkpoints. Each completed answer is saved before the next question runs. Keep RUN_NAME the same to resume an interrupted run with unchanged settings. Use a different RUN_NAME for a changed experiment.


In [ ]:
import torch
from google.colab import drive, userdata

assert torch.cuda.is_available(), "Choose a T4 GPU runtime before continuing."
hf_token = userdata.get("HF_TOKEN")
drive.mount("/content/drive")
RUN_NAME = "run_01"
output_dir = Path("/content/drive/MyDrive/llm-safety-audit") / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)
print("GPU", torch.cuda.get_device_name(0))
print("Checkpoint folder", output_dir)


## 4. Inspect the examples and expected behaviors

Read the prompts, labels, and rationales. The owner should decide whether each expected behavior fits the research question before inspecting model answers. Revise the dataset and rationale if you disagree. The safety instruction is fixed for this experiment. Development and held-out examples are identified in the data.


In [ ]:
raw_path = project / "data/phishing_audit.jsonl"
examples = [json.loads(line) for line in raw_path.read_text().splitlines() if line.strip()]
for example in examples:
    print(example["id"], example["split"], "expected", example["expected_behavior"])
    print(example["prompt"])
    print("Rationale", example["rationale"])
    print()
print((project / "REVIEW_GUIDE.md").read_text())


## 5. Run CuratorKIT data curation

The experiment uses CuratorKIT's exact deduplicator. It normalizes prompt text when checking for duplicates and records provenance. AuditKIT will later check the actual generated answers for refusal phrases. Its phrase detector is supplementary to human judgments.


In [ ]:
from audit_toolkits import curate_prompts

pins = {
    "CuratorKIT": "72838b28bc5a758318f88ba2713822bed746c94b",
    "AuditKIT": "9d4e21bac240e8108a22c2c137d179d6070868d5",
}
roots = {}
for name, revision in pins.items():
    root = workspace / name
    subprocess.run(["git", "clone", "--quiet", f"https://github.com/Lexsi-Labs/{name}.git", str(root)], check=True)
    subprocess.run(["git", "-C", str(root), "checkout", "--quiet", revision], check=True)
    roots[name] = root
curated, curation_metadata = curate_prompts(examples, roots["CuratorKIT"])
curated_path = output_dir / "curated_prompts.jsonl"
curated_text = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in curated)
if curated_path.exists() and curated_path.read_text() != curated_text:
    raise ValueError("Dataset changed. Choose a new RUN_NAME before continuing.")
curated_path.write_text(curated_text, encoding="utf-8")
(output_dir / "curation.json").write_text(json.dumps(curation_metadata, indent=2) + "\n")
print(json.dumps(curation_metadata, indent=2))


## 6. Generate the answers

Read the safety instruction printed below. Run the cell and keep the session active while it works. Model loading can take several minutes. Generation time depends on answer length and Colab resources. Each progress line confirms a saved answer. The same cell resumes completed work after an interruption when settings and software match.


In [ ]:
import shutil
from run_audit import SAFETY_INSTRUCTION, run

print("Intervention instruction", SAFETY_INSTRUCTION)
print("The experiment will generate", len(curated) * 2, "answers.")
source_dir = output_dir / "source"
source_dir.mkdir(exist_ok=True)
for name in ("run_audit.py", "scoring.py", "audit_toolkits.py", "REVIEW_GUIDE.md", "REPORT.md"):
    shutil.copy2(project / name, source_dir / name)
(output_dir / "code_revision.txt").write_text(code_revision + "\n")
run(curated_path, output_dir, hf_token, max_new_tokens=384)


## 7. Run AuditKIT and prepare the human review sheet

AuditKIT's phrase-based detector can miss refusals and can flag phrases used in other contexts. Read each response using the review guide to assign the primary judgments. The sheet preserves your existing reviews when rerun.


In [ ]:
from audit_toolkits import audit_rows
from scoring import prepare

results_path = output_dir / "results.jsonl"
results = [json.loads(line) for line in results_path.read_text().splitlines() if line.strip()]
expected_keys = {(row["id"], condition) for row in curated for condition in ("baseline", "intervention")}
actual_keys = {(row["id"], row["condition"]) for row in results}
if actual_keys != expected_keys or len(results) != len(expected_keys):
    raise ValueError("Generation is incomplete. Resume the generation cell before preparing reviews.")
annotated, audit_metadata = audit_rows(results, roots["AuditKIT"])
(output_dir / "auditkit_annotations.jsonl").write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in annotated), encoding="utf-8"
)
(output_dir / "toolkit_audit.json").write_text(json.dumps(audit_metadata, indent=2) + "\n")
review_path = output_dir / "reviews.csv"
if review_path.exists():
    print("Keeping existing review sheet", review_path)
else:
    prepare(results_path, review_path)
    print("Created review sheet", review_path)
print(json.dumps(audit_metadata, indent=2))


## 8. Download and fill the review sheet

Open reviews.csv in a spreadsheet application. Fill only judgment and reason for each of the 48 responses using REVIEW_GUIDE.md. Save it as UTF-8 CSV. You can also send the file back in chat for help interpreting ambiguous responses. Final judgments belong to the owner.


In [ ]:
from google.colab import files
files.download(str(review_path))


## 9. Upload the completed review and calculate metrics

Run this cell after completing the review. Choose exactly one CSV file. A pending report identifies judgments still needed. Rates with unclear or truncated cases are marked exploratory. The report includes counts for interpretation.


In [ ]:
from scoring import score
from run_audit import write_json

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choose exactly one completed review CSV.")
review_bytes = next(iter(uploaded.values()))
completed_review = output_dir / "completed_reviews.csv"
candidate_review = output_dir / "candidate_reviews.csv"
candidate_review.write_bytes(review_bytes)
report = score(results_path, candidate_review)
completed_review.write_bytes(review_bytes)
write_json(output_dir / "metrics.json", report)
print(json.dumps(report, indent=2))


## 10. Export artifacts

This bundle contains prompts, actual answers, software and model provenance, toolkit records, code, and review files. Upload its extracted files to your Hugging Face dataset repository. Use the actual metrics to complete REPORT.md and save your executed notebook to GitHub. The model remains the original checkpoint, with a system-instruction intervention applied during generation.


In [ ]:
bundle_path = shutil.make_archive(str(output_dir.parent / (RUN_NAME + "_artifacts")), "zip", output_dir)
files.download(bundle_path)
print("Artifacts also remain in", output_dir)
